[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB06_Aplicacion_Despliegue_Modelo.ipynb)
# NB06 — Aplicación y despliegue del modelo
**Correspondencia: Semanas 16–17**


## 1. Preparación del entorno


In [ ]:
!pip -q install streamlit onnxruntime

import os
import json
import shutil
from pathlib import Path

print("Entorno preparado")


## 2. Conectar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Recuperar los artefactos generados en NB05

Se utilizarán:

- `modelo_cnn.onnx`
- `class_names.json`

Ambos archivos deben corresponder al mismo modelo y conservar el mismo orden de clases utilizado durante el entrenamiento.


In [ ]:
SOURCE_DIR = Path(
    "/content/drive/MyDrive/MachineLearning2026/implementacion"
)

MODEL_SOURCE = SOURCE_DIR / "modelo_cnn.onnx"
CLASSES_SOURCE = SOURCE_DIR / "class_names.json"

print("Modelo:", MODEL_SOURCE.exists())
print("Clases:", CLASSES_SOURCE.exists())


## 4. Crear la estructura del proyecto

La aplicación que posteriormente se almacenará en GitHub tendrá una estructura mínima:

```text
ml-image-classifier/
├── app.py
├── modelo_cnn.onnx
├── class_names.json
├── requirements.txt
└── README.md
```


In [ ]:
PROJECT_DIR = Path("/content/ml-image-classifier")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(MODEL_SOURCE, PROJECT_DIR / "modelo_cnn.onnx")
shutil.copy(CLASSES_SOURCE, PROJECT_DIR / "class_names.json")

print("Proyecto creado en:")
print(PROJECT_DIR)


## 5. Verificar las clases


In [ ]:
with open(
    PROJECT_DIR / "class_names.json",
    "r",
    encoding="utf-8"
) as f:
    CLASS_NAMES = json.load(f)

for i, clase in enumerate(CLASS_NAMES):
    print(i, "→", clase)


## 6. Probar el modelo antes de construir la aplicación

Antes de crear la interfaz verificaremos que ONNX Runtime pueda cargar correctamente el modelo.


In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(
    str(PROJECT_DIR / "modelo_cnn.onnx"),
    providers=["CPUExecutionProvider"]
)

input_info = session.get_inputs()[0]
output_info = session.get_outputs()[0]

print("Input:", input_info.name)
print("Input shape:", input_info.shape)
print("Input type:", input_info.type)
print()
print("Output:", output_info.name)
print("Output shape:", output_info.shape)


## 7. Definir el pipeline de inferencia

La aplicación debe reproducir el mismo procesamiento utilizado durante el entrenamiento.

Para el modelo desarrollado con MobileNetV2:

**imagen → RGB → 224 × 224 → float32 → modelo → probabilidades → clase + confianza**


In [ ]:
import numpy as np
from PIL import Image

IMG_SIZE = (224, 224)

def preparar_imagen(imagen):
    imagen = imagen.convert("RGB")
    imagen = imagen.resize(IMG_SIZE)

    x = np.array(imagen, dtype=np.float32)
    x = np.expand_dims(x, axis=0)

    return x

def predecir(imagen):
    x = preparar_imagen(imagen)

    input_name = session.get_inputs()[0].name

    output = session.run(
        None,
        {input_name: x}
    )[0]

    indice = int(np.argmax(output[0]))
    confianza = float(np.max(output[0]))

    return CLASS_NAMES[indice], confianza, output[0]

print("Funciones creadas")


## 8. Probar inferencia con una imagen nueva


In [ ]:
IMAGE_PATH = (
    "/content/drive/MyDrive/MachineLearning2026/imagen_prueba.jpg"
)

imagen = Image.open(IMAGE_PATH)

clase, confianza, probabilidades = predecir(imagen)

print("Predicción:", clase)
print(f"Confianza: {confianza:.2%}")


## 9. Revisar probabilidades por clase


In [ ]:
import pandas as pd

resultado = pd.DataFrame({
    "Clase": CLASS_NAMES,
    "Probabilidad": probabilidades
}).sort_values(
    "Probabilidad",
    ascending=False
)

resultado


## 10. Construir la aplicación Streamlit

La aplicación permitirá:

1. seleccionar una imagen;
2. visualizarla;
3. ejecutar la inferencia;
4. mostrar la clase predicha;
5. mostrar el nivel de confianza;
6. visualizar las probabilidades por clase.


In [ ]:
app_code = r"""
import json
import numpy as np
import pandas as pd
import streamlit as st
import onnxruntime as ort
from PIL import Image

MODEL_PATH = "modelo_cnn.onnx"
CLASSES_PATH = "class_names.json"
IMG_SIZE = (224, 224)

st.set_page_config(
    page_title="Clasificador de imágenes",
    page_icon="🖼️"
)

@st.cache_resource
def cargar_modelo():
    return ort.InferenceSession(
        MODEL_PATH,
        providers=["CPUExecutionProvider"]
    )

@st.cache_data
def cargar_clases():
    with open(CLASSES_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

session = cargar_modelo()
class_names = cargar_clases()

def preparar_imagen(imagen):
    imagen = imagen.convert("RGB")
    imagen = imagen.resize(IMG_SIZE)

    x = np.array(imagen, dtype=np.float32)
    x = np.expand_dims(x, axis=0)

    return x

def predecir(imagen):
    x = preparar_imagen(imagen)

    input_name = session.get_inputs()[0].name

    output = session.run(
        None,
        {input_name: x}
    )[0]

    indice = int(np.argmax(output[0]))
    confianza = float(np.max(output[0]))

    return indice, confianza, output[0]

st.title("Clasificador de imágenes")

archivo = st.file_uploader(
    "Seleccione una imagen",
    type=["jpg", "jpeg", "png"]
)

if archivo is not None:
    imagen = Image.open(archivo)

    st.image(
        imagen,
        caption="Imagen seleccionada",
        use_container_width=True
    )

    if st.button("Clasificar"):
        indice, confianza, probabilidades = predecir(imagen)

        st.subheader("Resultado")
        st.write(f"**Clase predicha:** {class_names[indice]}")
        st.write(f"**Confianza:** {confianza:.2%}")

        if confianza < 0.60:
            st.warning(
                "La predicción presenta un nivel de confianza bajo."
            )

        tabla = pd.DataFrame({
            "Clase": class_names,
            "Probabilidad": probabilidades
        }).sort_values(
            "Probabilidad",
            ascending=False
        )

        st.subheader("Probabilidades por clase")
        st.dataframe(
            tabla,
            hide_index=True,
            use_container_width=True
        )
"""

with open(
    PROJECT_DIR / "app.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(app_code)

print("app.py creado")


## 11. Crear `requirements.txt`


In [ ]:
requirements = '''streamlit
onnxruntime
numpy
pandas
Pillow
'''

with open(
    PROJECT_DIR / "requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

print(requirements)


## 12. Crear `README.md`

El README documentará el propósito y los archivos mínimos del producto.


In [ ]:
readme = """# Clasificador de imágenes con Deep Learning

Aplicación desarrollada con Streamlit para ejecutar un modelo de clasificación de imágenes exportado a ONNX.

## Archivos

- `app.py`: aplicación Streamlit.
- `modelo_cnn.onnx`: modelo de clasificación.
- `class_names.json`: orden de las clases.
- `requirements.txt`: dependencias.

## Ejecución

```bash
streamlit run app.py
```

## Flujo

Imagen → preprocesamiento → ONNX Runtime → clase predicha → confianza
"""

with open(
    PROJECT_DIR / "README.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

print("README.md creado")


## 13. Revisar la estructura final


In [ ]:
for archivo in PROJECT_DIR.iterdir():
    tamaño = archivo.stat().st_size / (1024 ** 2)

    print(
        f"{archivo.name:25} {tamaño:.2f} MB"
    )


## 14. Validar los archivos requeridos

Antes del despliegue comprobaremos que el proyecto contenga todos los componentes mínimos.


In [ ]:
archivos_requeridos = [
    "app.py",
    "modelo_cnn.onnx",
    "class_names.json",
    "requirements.txt",
    "README.md"
]

for nombre in archivos_requeridos:
    ruta = PROJECT_DIR / nombre

    estado = "OK" if ruta.exists() else "FALTA"

    print(f"{estado:6} {nombre}")


## 15. Probar la aplicación en Google Colab

Esta ejecución permite comprobar que Streamlit inicia correctamente.

Google Colab se utiliza aquí únicamente como entorno de prueba. El producto final será desplegado posteriormente mediante Streamlit Community Cloud.


In [ ]:
!streamlit run /content/ml-image-classifier/app.py --server.headless true &> /content/streamlit.log &

import time
time.sleep(5)

print(open("/content/streamlit.log").read())


## 16. Preparar el proyecto para GitHub

El repositorio deberá contener como mínimo:

```text
app.py
modelo_cnn.onnx
class_names.json
requirements.txt
README.md
```

El repositorio será la fuente utilizada por Streamlit Community Cloud para construir y desplegar la aplicación.


## 17. Crear el repositorio en GitHub

Desde el navegador:

1. Ingrese a GitHub.
2. Cree un nuevo repositorio.
3. Asigne un nombre al proyecto.
4. Incorpore los archivos generados en este notebook.
5. Verifique que `app.py` y `requirements.txt` se encuentren en el repositorio.
6. Compruebe que el modelo y `class_names.json` correspondan a la versión definitiva.

Una estructura mínima será:

```text
repositorio/
├── app.py
├── modelo_cnn.onnx
├── class_names.json
├── requirements.txt
└── README.md
```


## 18. Despliegue con Streamlit Community Cloud

Desde el navegador:

1. Ingrese a Streamlit Community Cloud.
2. Conecte su cuenta de GitHub.
3. Seleccione el repositorio del proyecto.
4. Seleccione la rama correspondiente.
5. Indique `app.py` como archivo principal.
6. Inicie el despliegue.
7. Espere la instalación de las dependencias.
8. Compruebe que la aplicación se inicia correctamente.

Al finalizar se obtendrá una **URL pública** para acceder a la aplicación.


## 19. Flujo completo de despliegue

```text
Google Colab
     ↓
modelo CNN
     ↓
ONNX
     ↓
app.py + requirements.txt
     ↓
GitHub
     ↓
Streamlit Community Cloud
     ↓
URL pública
     ↓
Usuario carga imagen
     ↓
Inferencia
     ↓
Clase + confianza
```


## 20. Pruebas funcionales

Una vez desplegada la aplicación, pruebe al menos:

- una imagen válida de cada clase;
- una imagen difícil;
- una imagen con baja confianza;
- un archivo no permitido;
- una imagen diferente a las utilizadas durante entrenamiento.

Registre el comportamiento observado.


## 21. Tabla de pruebas

Complete la siguiente tabla con los resultados reales de la aplicación desplegada.


In [ ]:
pruebas = pd.DataFrame({
    "Prueba": [
        "Imagen clase 1",
        "Imagen clase 2",
        "Imagen difícil",
        "Baja confianza",
        "Archivo inválido"
    ],
    "Resultado_esperado": [
        "",
        "",
        "",
        "",
        ""
    ],
    "Resultado_obtenido": [
        "",
        "",
        "",
        "",
        ""
    ],
    "Estado": [
        "",
        "",
        "",
        "",
        ""
    ]
})

pruebas


## 22. Versionado del producto

Utilice versiones identificables para evitar confusión entre artefactos.

Ejemplo:

```text
Modelo:
v1.0 — CNN definitiva
v1.1 — CNN optimizada
v1.2 — ONNX utilizado en producción

Aplicación:
v1.0 — primera versión desplegada
v1.1 — corrección de interfaz
```

El repositorio GitHub permite registrar los cambios realizados en el código y mantener trazabilidad del producto.


## 23. Integración continua

En este proyecto se utilizará un flujo simplificado:

```text
Cambio en el proyecto
        ↓
Actualización en GitHub
        ↓
Streamlit Community Cloud
        ↓
Reconstrucción / actualización
        ↓
Aplicación desplegada
```

Antes de incorporar una nueva versión del modelo deberían verificarse, como mínimo:

- carga correcta del modelo;
- forma de entrada;
- número y orden de clases;
- ejecución de inferencia;
- desempeño mínimo definido;
- latencia aceptable;
- funcionamiento de la aplicación.


## 24. Propuesta de criterios de validación

Defina criterios concretos para aceptar una nueva versión del producto.

Ejemplo:

```text
Accuracy test >= 85%
Ninguna clase con Recall < 70%
Modelo carga sin errores
Input = 224 × 224 × 3
Output = número esperado de clases
Aplicación inicia correctamente
Inferencia funcional
```

Los valores definitivos deben establecerse de acuerdo con las características del proyecto.


## 25. Cloud y Edge Computing

La aplicación desplegada mediante Streamlit Community Cloud representa una implementación **cloud**:

```text
Usuario
   ↓
Internet
   ↓
Aplicación Streamlit
   ↓
ONNX Runtime
   ↓
Modelo
   ↓
Predicción
```

Como alternativa conceptual, el mismo modelo podría ejecutarse cerca de la fuente de datos mediante **Edge Computing**, siempre que el dispositivo disponga de los recursos necesarios.

La elección entre cloud, local y Edge depende de factores como:

- conectividad;
- latencia;
- privacidad;
- recursos disponibles;
- costo;
- facilidad de actualización.


## 26. Actividad

A partir del modelo desarrollado en el proyecto integrador:

1. Recupere el modelo ONNX generado en NB05.
2. Verifique el orden de las clases.
3. Ejecute una inferencia mediante ONNX Runtime.
4. Construya `app.py` con Streamlit.
5. Incorpore carga y visualización de imágenes.
6. Muestre clase predicha y nivel de confianza.
7. Incorpore manejo básico de baja confianza.
8. Genere `requirements.txt`.
9. Documente el proyecto mediante `README.md`.
10. Organice todos los artefactos en un repositorio GitHub.
11. Despliegue la aplicación mediante Streamlit Community Cloud.
12. Obtenga y compruebe la URL pública.
13. Ejecute pruebas funcionales con imágenes nuevas.
14. Registre las versiones del modelo y de la aplicación.
15. Defina criterios mínimos para aceptar futuras actualizaciones.


## 27. Base para la Evaluación 4

Este notebook completa el flujo práctico del proyecto integrador.

El resultado esperado es una **aplicación funcional y desplegada**, capaz de recibir imágenes nuevas y ejecutar inferencias utilizando el modelo desarrollado durante el semestre.

**Modelo CNN → optimización → ONNX → Streamlit → GitHub → Streamlit Community Cloud → URL funcional → imagen nueva → clase + confianza**

La aplicación desplegada, junto con la evidencia técnica generada durante NB05 y NB06, constituye la base para la presentación del **producto final operativo** en la Evaluación 4.
